In [ ]:
from esm.models.esmc import EsmcForMaskedLM, EsmcTokenizer
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# ESMC-300M Pretrained model
model = EsmcForMaskedLM.from_pretrained(
    "biohub/ESMC-300M",
    device=device,
).eval()

tokenizer = EsmcTokenizer()

In [ ]:
def get_protein_embedding(amino_acid_seq):

    inputs = tokenizer([amino_acid_seq], return_tensors="pt", padding=True)
    inputs = {name: value.to(model.device) for name, value in inputs.items()}

    with torch.inference_mode():
        result = model(**inputs, output_hidden_states=True)

    token_embeddings = result.hidden_states[-1]    
    residue_embeddings = token_embeddings[0, 1:-1] # exlude start and end token

    return residue_embeddings

def get_sDBM_embeds(df):
    sDBM_tensor_list = []

    for index, row in df.iterrows():
        sDBM_start, sDBM_end = ast.literal_eval(row.values[0])
        sDBM_end += 1

        amino_seq = row.values[3]

        sDBM_embedding = get_protein_embedding(amino_seq)[sDBM_start : sDBM_end, :]

    # accumulating embeddings on the gpu -> save them on cpu instead
        sDBM_tensor_list.append(sDBM_embedding.cpu())

    embedding_tensor = torch.stack(sDBM_tensor_list, dim = 0)

    print(embedding_tensor.shape)

    return embedding_tensor


def get_dna_encoding(ori_seq):

    dna_vec_list = []

    for base in ori_seq:
        match base:
            case 'A':
                dna_vec_list.append(0)
            case 'C':
                dna_vec_list.append(1)
            case 'G':
                dna_vec_list.append(2)
            case 'T':  
                dna_vec_list.append(3)

    #convert to tensor
    dna_encoding = torch.tensor(dna_vec_list, dtype=torch.long)

    return dna_encoding


def get_ori_encoding(df):
    ori_tensor_list = []

    for index, row in df.iterrows():

        ori_seq = row.values[2]

        dna_encoding = get_dna_encoding(ori_seq)


    # accumulating embeddings on the gpu -> save them on cpu instead
        ori_tensor_list.append(dna_encoding.unsqueeze(dim=0))


    ori_encoding_tensor = torch.stack(ori_tensor_list, dim = 0)

    ori_encoding = ori_encoding_tensor.flatten(-2)

    return ori_encoding

In [ ]:
from pathlib import Path
import torch


def build_and_save_tensors(
    df_rep,
    negative_rep,
    df_relax,
    negative_relax,
    get_sDBM_embeds,
    get_ori_encoding,
    apply_PCA,
    save_path,
    train_fraction=0.7,
    seed=42,
):
    def split_df(df, split_seed):
        generator = torch.Generator().manual_seed(split_seed)
        order = torch.randperm(len(df), generator=generator).tolist()
        split_at = int(len(df) * train_fraction)
        return df.iloc[order[:split_at]].copy(), df.iloc[order[split_at:]].copy()

    rep_train, rep_val = split_df(df_rep, seed)
    neg_rep_train, neg_rep_val = split_df(negative_rep, seed + 1)
    relax_train, relax_val = split_df(df_relax, seed + 2)
    neg_relax_train, neg_relax_val = split_df(negative_relax, seed + 3)

    # Keep groups in a fixed order so the reduced tensor can be split back correctly.
    groups = [
        rep_train,
        neg_rep_train,
        relax_train,
        neg_relax_train,
        rep_val,
        neg_rep_val,
        relax_val,
        neg_relax_val,
    ]

    # Concatenate all protein embeddings, apply PCA once, then restore the groups.
    raw_proteins = torch.cat(
        [get_sDBM_embeds(group).detach().cpu() for group in groups],
        dim=0,
    )
    reduced_proteins = apply_PCA(raw_proteins)

    group_sizes = [len(group) for group in groups]
    reduced_groups = torch.split(reduced_proteins, group_sizes, dim=0)

    def add_dna(reduced_protein, df):
        dna = get_ori_encoding(df).detach().cpu().float()
        reduced_protein = reduced_protein.detach().cpu().float()

        if reduced_protein.shape[0] != dna.shape[0]:
            raise ValueError("Protein and DNA embeddings have different row counts.")

        return torch.cat((reduced_protein, dna), dim=1)

    train_groups = groups[:4]
    train_reduced = reduced_groups[:4]

    X_train = torch.cat(
        [add_dna(protein, df) for protein, df in zip(train_reduced, train_groups)],
        dim=0,
    )
    y_train = torch.cat([
        torch.full((len(rep_train),), 1, dtype=torch.long),
        torch.full((len(neg_rep_train),), 0, dtype=torch.long),
        torch.full((len(relax_train),), 1, dtype=torch.long),
        torch.full((len(neg_relax_train),), 0, dtype=torch.long),
    ])

    # Shuffle training features and labels together.
    generator = torch.Generator().manual_seed(seed)
    order = torch.randperm(len(y_train), generator=generator)
    X_train, y_train = X_train[order], y_train[order]

    X_val_rep = torch.cat((
        add_dna(reduced_groups[4], rep_val),
        add_dna(reduced_groups[5], neg_rep_val),
    ))
    y_val_rep = torch.cat((
        torch.ones(len(rep_val), dtype=torch.long),
        torch.zeros(len(neg_rep_val), dtype=torch.long),
    ))

    X_val_relax = torch.cat((
        add_dna(reduced_groups[6], relax_val),
        add_dna(reduced_groups[7], neg_relax_val),
    ))
    y_val_relax = torch.cat((
        torch.ones(len(relax_val), dtype=torch.long),
        torch.zeros(len(neg_relax_val), dtype=torch.long),
    ))

    tensors = {
        "X_train": X_train,
        "y_train": y_train,
        "X_val_rep": X_val_rep,
        "y_val_rep": y_val_rep,
        "X_val_relax": X_val_relax,
        "y_val_relax": y_val_relax,
    }

    save_dir = Path(save_path)
    save_dir.mkdir(parents=True, exist_ok=True)

    for name, tensor in tensors.items():
        torch.save(tensor, save_dir / f"{name}.pt")

    return tensors